# TOB予測ポートフォリオ バックテスト 2022-2025

毎年5月末にRFモデルで予測確率上位N%銘柄を等ウェイトでロング、翌5月末まで保持。TOPIX比較。

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CREDENTIALS_PATH = PROJECT_ROOT / "keys" / "gcp-service-account.json"
BQ_PROJECT = "gmailpj-357912"
CACHE_DIR = Path("C:/tmp/tob_prediction")

EVAL_YEARS = [2022, 2023, 2024, 2025]
TOP_PCTS = [0.05, 0.15, 0.25]
TRANSACTION_COST = 0.001  # 片道10bp（往復20bp）

JST = ZoneInfo("Asia/Tokyo")

creds = service_account.Credentials.from_service_account_file(
    str(CREDENTIALS_PATH),
    scopes=["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project=BQ_PROJECT, credentials=creds)
print("BQ client ready")

## 1. 予測確率の読み込み

In [ ]:
preds_all = []
for year in EVAL_YEARS:
    p = CACHE_DIR / f"predictions_{year}.csv"
    if not p.exists():
        raise FileNotFoundError(f"predictions_{year}.csv not found. Run train_rf.py first.")
    df = pd.read_csv(p, encoding="utf-8", dtype={"TICKER": str})
    df["year"] = year
    preds_all.append(df)

preds = pd.concat(preds_all, ignore_index=True)
print(f"Predictions loaded: {len(preds)} rows, years={sorted(preds['year'].unique())}")
preds.groupby("year").agg(n=("TICKER", "count"), pos=("label", "sum"), prob_mean=("prob", "mean")).round(4)

## 2. 株価データ取得（6月初〜翌5月末）

In [ ]:
min_year = min(EVAL_YEARS)
max_year = max(EVAL_YEARS)

price_cache = CACHE_DIR / "bt_prices.csv"
if price_cache.exists():
    prices = pd.read_csv(price_cache, encoding="utf-8", dtype={"TICKER": str}, parse_dates=["DATE"])
    print(f"Price cache loaded: {len(prices)} rows")
else:
    sql = f"""
    SELECT TICKER, DATE, ADJ_CLOSE
    FROM `gmailpj-357912.STOCK.STOCK_PRICE_JQUANTS`
    WHERE DATE >= '{min_year}-05-20' AND DATE <= '{max_year + 1}-06-05'
      AND IS_PREFERRED = FALSE
      AND ADJ_CLOSE IS NOT NULL AND ADJ_CLOSE > 0
    ORDER BY TICKER, DATE
    """
    prices = client.query(sql).to_dataframe()
    prices["TICKER"] = prices["TICKER"].astype(str)
    prices.to_csv(price_cache, index=False, encoding="utf-8")
    print(f"Prices from BQ: {len(prices)} rows")

print(f"Date range: {prices['DATE'].min()} to {prices['DATE'].max()}")
print(f"Tickers: {prices['TICKER'].nunique()}")

## 3. TOPIX取得

In [ ]:
topix_cache = CACHE_DIR / "bt_topix.csv"
if topix_cache.exists():
    topix = pd.read_csv(topix_cache, encoding="utf-8", parse_dates=["DATE"])
    print(f"TOPIX cache loaded: {len(topix)} rows")
else:
    sql_topix = f"""
    SELECT DATE, CLOSE
    FROM `gmailpj-357912.STOCK.INDEX_PRICE`
    WHERE INDEX_CODE = '0000'
      AND DATE >= '{min_year}-05-20' AND DATE <= '{max_year + 1}-06-05'
    ORDER BY DATE
    """
    topix = client.query(sql_topix).to_dataframe()
    topix.to_csv(topix_cache, index=False, encoding="utf-8")
    print(f"TOPIX from BQ: {len(topix)} rows")

topix = topix.sort_values("DATE").reset_index(drop=True)
topix.tail()

## 4. ポートフォリオ構築関数

In [ ]:
def find_ref_date(dates: pd.Series, year: int, month: int, last: bool = True) -> pd.Timestamp:
    """指定月の最初/最後の営業日を取得."""
    m_dates = dates[(dates.dt.year == year) & (dates.dt.month == month)]
    if m_dates.empty:
        m_dates = dates[(dates.dt.year == year) & (dates.dt.month == month - 1)]
    return m_dates.max() if last else m_dates.min()


def compute_annual_return(
    tickers: list[str],
    entry_date: pd.Timestamp,
    exit_date: pd.Timestamp,
    price_df: pd.DataFrame,
) -> dict:
    """等ウェイトポートフォリオの年間リターンを計算."""
    rets = []
    details = []
    for tk in tickers:
        tk_prices = price_df[price_df["TICKER"] == tk].set_index("DATE")["ADJ_CLOSE"].sort_index()
        if tk_prices.empty:
            continue
        # entry: entry_date以降の最初の株価
        entry_prices = tk_prices[tk_prices.index >= entry_date]
        if entry_prices.empty:
            continue
        p_entry = entry_prices.iloc[0]
        # exit: exit_date以前の最後の株価（上場廃止の場合は最後の取引価格）
        exit_prices = tk_prices[tk_prices.index <= exit_date]
        if exit_prices.empty:
            continue
        p_exit = exit_prices.iloc[-1]
        ret = (p_exit / p_entry) - 1
        rets.append(ret)
        details.append({"ticker": tk, "entry": p_entry, "exit": p_exit, "ret": ret})
    
    if not rets:
        return {"port_ret": np.nan, "n_stocks": 0, "details": []}
    
    avg_ret = np.mean(rets)
    # 往復取引コスト
    net_ret = avg_ret - 2 * TRANSACTION_COST
    return {"port_ret": net_ret, "n_stocks": len(rets), "details": details}


def compute_topix_return(entry_date: pd.Timestamp, exit_date: pd.Timestamp) -> float:
    """TOPIX年間リターン."""
    entry_prices = topix[topix["DATE"] >= entry_date]
    exit_prices = topix[topix["DATE"] <= exit_date]
    if entry_prices.empty or exit_prices.empty:
        return np.nan
    return (exit_prices["CLOSE"].iloc[-1] / entry_prices["CLOSE"].iloc[0]) - 1


print("Functions defined")

## 5. バックテスト実行

In [ ]:
all_dates = prices["DATE"].drop_duplicates().sort_values()

results = []
for year in EVAL_YEARS:
    entry_date = find_ref_date(all_dates, year, 6, last=False)  # 6月初
    exit_date = find_ref_date(all_dates, year + 1, 5, last=True)  # 翌5月末
    
    year_preds = preds[preds["year"] == year].sort_values("prob", ascending=False)
    n_total = len(year_preds)
    
    topix_ret = compute_topix_return(entry_date, exit_date)
    
    for pct in TOP_PCTS:
        n_select = max(1, int(n_total * pct))
        selected = year_preds.head(n_select)["TICKER"].tolist()
        
        res = compute_annual_return(selected, entry_date, exit_date, prices)
        
        results.append({
            "year": year,
            "top_pct": f"Top {int(pct*100)}%",
            "n_selected": n_select,
            "n_traded": res["n_stocks"],
            "port_ret": res["port_ret"],
            "topix_ret": topix_ret,
            "alpha": res["port_ret"] - topix_ret if not np.isnan(res["port_ret"]) else np.nan,
            "entry_date": entry_date,
            "exit_date": exit_date,
        })
        print(f"{year} {f'Top {int(pct*100)}%':>8s}: n={res['n_stocks']}, ret={res['port_ret']:+.1%}, TOPIX={topix_ret:+.1%}, alpha={res['port_ret']-topix_ret:+.1%}")

bt = pd.DataFrame(results)
bt

## 6. サマリー統計

In [ ]:
summary = bt.groupby("top_pct").agg(
    avg_ret=("port_ret", "mean"),
    avg_topix=("topix_ret", "mean"),
    avg_alpha=("alpha", "mean"),
    win_years=("alpha", lambda x: (x > 0).sum()),
    total_years=("alpha", "count"),
).round(4)

# 累積リターン（複利）
for pct_label in [f"Top {int(p*100)}%" for p in TOP_PCTS]:
    rets = bt[bt["top_pct"] == pct_label]["port_ret"].values
    cum = np.prod(1 + rets) - 1
    summary.loc[pct_label, "cum_ret"] = round(cum, 4)

topix_cum = np.prod(1 + bt.groupby("year")["topix_ret"].first().values) - 1
summary["topix_cum"] = round(topix_cum, 4)

print("=== TOB予測ポートフォリオ バックテスト 2022-2025 ===")
print(f"取引コスト: 往復{TRANSACTION_COST*2*100:.0f}bp")
print()
summary

## 7. 日次エクイティカーブ（詳細分析用）

In [ ]:
def build_daily_equity(
    year: int,
    tickers: list[str],
    entry_date: pd.Timestamp,
    exit_date: pd.Timestamp,
    price_df: pd.DataFrame,
) -> pd.DataFrame:
    """等ウェイトポートフォリオの日次エクイティを構築."""
    mask = (price_df["DATE"] >= entry_date) & (price_df["DATE"] <= exit_date)
    period_prices = price_df[mask & price_df["TICKER"].isin(tickers)].copy()
    
    pivot = period_prices.pivot_table(index="DATE", columns="TICKER", values="ADJ_CLOSE")
    pivot = pivot.sort_index()
    
    # 各銘柄を初日=1に正規化
    normalized = pivot / pivot.iloc[0]
    # 等ウェイト平均
    equity = normalized.mean(axis=1)
    return equity.to_frame(name="equity")


# 全年度つなぎ合わせ
equity_curves = {}
for pct in TOP_PCTS:
    label = f"Top {int(pct*100)}%"
    yearly_eq = []
    cum_val = 1.0
    for year in EVAL_YEARS:
        row = bt[(bt["year"] == year) & (bt["top_pct"] == label)].iloc[0]
        year_preds_sorted = preds[preds["year"] == year].sort_values("prob", ascending=False)
        n_select = max(1, int(len(year_preds_sorted) * pct))
        selected = year_preds_sorted.head(n_select)["TICKER"].tolist()
        
        eq = build_daily_equity(year, selected, row["entry_date"], row["exit_date"], prices)
        # 取引コスト控除（entry時）
        eq["equity"] = eq["equity"] * (1 - TRANSACTION_COST)
        # 前年の累積に接続
        eq["equity_cum"] = eq["equity"] * cum_val
        yearly_eq.append(eq[["equity_cum"]])
        cum_val = eq["equity_cum"].iloc[-1] * (1 - TRANSACTION_COST)  # exit時コスト
    
    curve = pd.concat(yearly_eq)
    equity_curves[label] = curve

# TOPIX
topix_eq = []
cum_val_topix = 1.0
for year in EVAL_YEARS:
    row = bt[bt["year"] == year].iloc[0]
    t_period = topix[(topix["DATE"] >= row["entry_date"]) & (topix["DATE"] <= row["exit_date"])].copy()
    t_period = t_period.set_index("DATE").sort_index()
    t_period["equity_cum"] = (t_period["CLOSE"] / t_period["CLOSE"].iloc[0]) * cum_val_topix
    topix_eq.append(t_period[["equity_cum"]])
    cum_val_topix = t_period["equity_cum"].iloc[-1]

topix_curve = pd.concat(topix_eq)
equity_curves["TOPIX"] = topix_curve

print(f"Equity curves built: {list(equity_curves.keys())}")

## 8. エクイティカーブ描画

In [ ]:
try:
    import japanize_matplotlib
except ImportError:
    pass

fig, ax = plt.subplots(figsize=(14, 7))

colors = {"Top 5%": "#e74c3c", "Top 15%": "#3498db", "Top 25%": "#2ecc71", "TOPIX": "#7f8c8d"}
linewidths = {"Top 5%": 2.5, "Top 15%": 2.0, "Top 25%": 1.5, "TOPIX": 2.0}
linestyles = {"Top 5%": "-", "Top 15%": "-", "Top 25%": "-", "TOPIX": "--"}

for label, curve in equity_curves.items():
    final_val = curve["equity_cum"].iloc[-1]
    ax.plot(
        curve.index, curve["equity_cum"],
        label=f"{label} ({final_val:.2f}x)",
        color=colors.get(label, "gray"),
        linewidth=linewidths.get(label, 1.5),
        linestyle=linestyles.get(label, "-"),
    )

# リバランス境界
for year in EVAL_YEARS[1:]:
    rebal_date = find_ref_date(all_dates, year, 6, last=False)
    ax.axvline(rebal_date, color="gray", linestyle=":", alpha=0.4)

ax.axhline(1.0, color="black", linestyle="-", alpha=0.2)
ax.set_title(f"TOB予測ポートフォリオ vs TOPIX ({min(EVAL_YEARS)}-{max(EVAL_YEARS)+1})", fontsize=14)
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return (1.0 = initial)")
ax.legend(loc="upper left", fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 9. MaxDD・Sharpe計算

In [ ]:
def calc_metrics(equity_series: pd.Series) -> dict:
    """日次エクイティカーブからMaxDD, Sharpe等を計算."""
    daily_ret = equity_series.pct_change().dropna()
    
    # MaxDD
    peak = equity_series.cummax()
    dd = (equity_series - peak) / peak
    max_dd = dd.min()
    
    # Sharpe (年率化: 250営業日)
    ann_ret = (equity_series.iloc[-1] / equity_series.iloc[0]) ** (250 / len(daily_ret)) - 1
    ann_vol = daily_ret.std() * np.sqrt(250)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    
    return {
        "total_ret": equity_series.iloc[-1] / equity_series.iloc[0] - 1,
        "ann_ret": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_dd": max_dd,
    }


metrics_rows = []
for label, curve in equity_curves.items():
    m = calc_metrics(curve["equity_cum"])
    m["portfolio"] = label
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index("portfolio")
metrics_df = metrics_df[["total_ret", "ann_ret", "ann_vol", "sharpe", "max_dd"]]
metrics_df.columns = ["Total Return", "Ann. Return", "Ann. Vol", "Sharpe", "Max DD"]

print("=== ポートフォリオ評価指標 ===")
metrics_df.style.format({
    "Total Return": "{:+.1%}",
    "Ann. Return": "{:+.1%}",
    "Ann. Vol": "{:.1%}",
    "Sharpe": "{:.2f}",
    "Max DD": "{:.1%}",
})

## 10. 年度別リターン比較チャート

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(EVAL_YEARS))
width = 0.2

for i, pct_label in enumerate([f"Top {int(p*100)}%" for p in TOP_PCTS]):
    rets = bt[bt["top_pct"] == pct_label]["port_ret"].values
    ax.bar(x + i * width, rets * 100, width, label=pct_label, color=list(colors.values())[i])

topix_rets = bt.groupby("year")["topix_ret"].first().values
ax.bar(x + 3 * width, topix_rets * 100, width, label="TOPIX", color=colors["TOPIX"], alpha=0.7)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels([f"{y}/{y+1}" for y in EVAL_YEARS])
ax.set_ylabel("Annual Return (%)")
ax.set_title("年度別リターン: TOB予測ポートフォリオ vs TOPIX")
ax.legend()
ax.axhline(0, color="black", linewidth=0.5)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## 11. TOBヒット分析（実際にTOBが発生した銘柄の貢献）

In [ ]:
hit_analysis = []
for year in EVAL_YEARS:
    for pct in TOP_PCTS:
        label = f"Top {int(pct*100)}%"
        year_preds_sorted = preds[preds["year"] == year].sort_values("prob", ascending=False)
        n_select = max(1, int(len(year_preds_sorted) * pct))
        selected = year_preds_sorted.head(n_select)
        
        n_tob = int(selected["label"].sum())
        n_total = len(selected)
        
        # 全銘柄中のTOB率
        base_rate = year_preds_sorted["label"].mean()
        select_rate = selected["label"].mean()
        lift = select_rate / base_rate if base_rate > 0 else np.nan
        
        hit_analysis.append({
            "year": year,
            "portfolio": label,
            "n_selected": n_total,
            "n_tob_hit": n_tob,
            "hit_rate": select_rate,
            "base_rate": base_rate,
            "lift": lift,
        })

hit_df = pd.DataFrame(hit_analysis)
print("=== TOBヒット分析 ===")
hit_df.style.format({
    "hit_rate": "{:.1%}",
    "base_rate": "{:.1%}",
    "lift": "{:.1f}x",
})

## 12. 判定

In [ ]:
# PASS/FAIL判定
top5_metrics = metrics_df.loc["Top 5%"]
topix_metrics = metrics_df.loc["TOPIX"]

checks = {
    "Top5% > TOPIX (累積)": top5_metrics["Total Return"] > topix_metrics["Total Return"],
    "Top5% Sharpe > 0.5": top5_metrics["Sharpe"] > 0.5,
    "Top5% MaxDD < 30%": abs(top5_metrics["Max DD"]) < 0.30,
    "Alpha正の年が過半数": int(summary.loc["Top 5%", "win_years"]) > len(EVAL_YEARS) / 2,
}

print("=== PASS/FAIL チェック ===")
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")

overall = "BACKTEST_PASS" if all(checks.values()) else "BACKTEST_FAIL"
print(f"\n総合判定: {overall}")